# Daltix Case — Data Discovery & Semantic Assessment

**Checkpoint 1 · In progress**

Understand the source data, assess its quality and test its meaning before defining cleaning rules or an analytical model.

## Contents

1. [Setup & Connection](#1-setup--connection)
2. [Source Table Overview](#2-source-table-overview)
3. [Local Raw Extraction](#3-local-raw-extraction)
4. [Automated Source Profiling](#4-automated-source-profiling)
5. [Initial Profiling Conclusions](#5-initial-profiling-conclusions)
6. [Weekly Prices — Semantic Assessment](#6-weekly-prices--semantic-assessment)
7. [Next Session](#7-next-session)

**Current position:** all seven tables have been extracted and profiled. Weekly-price grain and aggregate temporal coverage have been assessed. Promotion logic, price validity, dimension keys and joins remain open.

**Tomorrow:** select the project kernel and keep `RUN_SOURCE_DISCOVERY = False` and `RUN_EXTRACTION = False`. With the existing raw files, a fresh **Run All** uses local data, validates its manifest and runs the assessment without contacting PostgreSQL. Source credentials are not needed in this mode. The local DuckDB connection is closed at the end; rerun its initialization if continuing interactively after cleanup.

Analytical outputs are retained from earlier work. Source/export cells whose behavior changed had their stale outputs cleared. No data-cleaning rule or analytical join has been introduced.


## 1. Setup & Connection

Prepare a local session. Source access is optional and requires explicit configuration switches.

### 1.1 Imports

DuckDB runs the analytical SQL and Polars displays results. Small reusable helpers in `src/daltix_case/source_io.py` handle source connections, guarded extraction and local manifest validation. These helpers do not clean data or define analytical metrics.


In [ ]:
from pathlib import Path

import duckdb
import polars as pl
from dotenv import load_dotenv

from daltix_case.source_io import (
    TABLES,
    extract_snapshot,
    source_config,
    source_connection,
    validate_local_snapshot,
)

### 1.2 Project Configuration

Resolve paths from the repository root or `notebooks` directory and define the weekly-price file explicitly. Both source-discovery and extraction switches default to `False`.

Only explicit source access loads `.env` and checks the six connection variables. Existing local data can be assessed without database credentials. An enabled extraction requires a separate empty destination if a snapshot already exists.


In [ ]:
# Default to local analysis: neither switch enables remote work implicitly.
RUN_SOURCE_DISCOVERY = False
RUN_EXTRACTION = False

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
weekly_prices_path = (RAW_DIR / "weekly_prices.parquet").as_posix()

DB_SCHEMA = None
if RUN_SOURCE_DISCOVERY or RUN_EXTRACTION:
    load_dotenv(PROJECT_ROOT / ".env", override=True)
    DB_SCHEMA = source_config()["schema"]

print(
    "Local analysis mode"
    if not (RUN_SOURCE_DISCOVERY or RUN_EXTRACTION)
    else "Source access enabled"
)

### 1.3 Optional PostgreSQL Connection

When source discovery is enabled, request database identity through a short-lived Psycopg connection. The helper enforces read-only transactions, a 10-second connection timeout and a 30-second statement timeout. Context managers close the connection and cursor on success or failure.

Otherwise, this cell makes no source request.


In [ ]:
# Query source identity only on explicit opt-in; always close the connection.
connection_info = None
if RUN_SOURCE_DISCOVERY:
    with source_connection() as conn, conn.cursor() as cur:
        cur.execute("""
                SELECT
                    current_database() AS database,
                    current_user AS user,
                    version() AS postgres_version;
            """)
        connection_info = cur.fetchone()
else:
    print("Source connection check skipped.")

### 1.4 Connection Check Result

Display the identity result only if the optional source check ran. `None` is expected in local mode. Connection metadata should be reviewed before saving outputs for publication.


In [ ]:
# Connection identity is available only when source discovery was requested.
connection_info

## 2. Source Table Overview

**Question:** how large are the source tables, and where should analytical processing run?

With `RUN_SOURCE_DISCOVERY = True`, query PostgreSQL catalog estimates using a parameterized schema filter. The connection closes after the query. `estimated_rows` is approximate; table, index and total sizes measure different storage components.

This step established weekly prices as the largest source during the original discovery. It is skipped in normal local mode; exact local file counts appear in section 3.3.


In [ ]:
# Inspect catalog estimates only when source discovery is explicitly enabled.
source_table_scale = []
if RUN_SOURCE_DISCOVERY:
    with source_connection() as conn, conn.cursor() as cur:
        cur.execute(
            """
                SELECT
                    c.relname AS table_name,
                    c.reltuples::bigint AS estimated_rows,
                    pg_size_pretty(pg_relation_size(c.oid)) AS table_size,
                    pg_size_pretty(pg_indexes_size(c.oid)) AS index_size,
                    pg_size_pretty(pg_total_relation_size(c.oid)) AS total_size
                FROM pg_class AS c
                JOIN pg_namespace AS n ON n.oid = c.relnamespace
                WHERE n.nspname = %s AND c.relkind = 'r'
                ORDER BY c.relname;
                """,
            (DB_SCHEMA,),
        )
        source_table_scale = cur.fetchall()
    source_table_scale = pl.DataFrame(source_table_scale)
else:
    print("Remote source overview skipped; use the local manifest below.")

source_table_scale

## 3. Local Raw Extraction

The ELT workflow extracts the seven source tables to raw Parquet, then assesses and transforms data locally. Source values are preserved without cleaning or deduplication.

Extraction is optional. When enabled, the helper rejects existing destinations, exports to temporary staging, compares DuckDB's returned COPY row counts with Parquet metadata and publishes the files with a manifest. No extra remote COUNT scans are required for this export reconciliation.

This validates exported row counts and local file integrity; it does not guarantee a transactionally consistent snapshot across all seven source tables or certify business correctness.


### 3.1 Local DuckDB Session

Create the local analytical connection. It requires no PostgreSQL extension or database credentials. When extraction is explicitly enabled, the helper creates a separate read-only source attachment and closes it automatically.

The helper uses Psycopg's connection-string builder, SQL literal escaping and quoted schema/table identifiers. No schema preview is hard-coded, and credentials are not written into the manifest.


In [ ]:
# Local analytical connection; no extension installation or remote attachment.
duck = duckdb.connect()

### 3.2 Optional Source Extraction

With `RUN_EXTRACTION = False`, the following cell skips export. With `True`, it extracts the seven tables as ZSTD-compressed Parquet only into an unused snapshot destination.

All expected raw paths and the manifest path are checked before opening the source. Files are staged and checked before publication, and existing files are never overwritten by this helper. Interrupted source exports do not publish a successful manifest. An abrupt process or machine failure during final publication can still require manual investigation of an incomplete destination.


In [ ]:
# Export is opt-in and refuses to replace any existing snapshot file.
extraction_manifest = extract_snapshot(RAW_DIR, enabled=RUN_EXTRACTION)
print(
    "Extraction completed."
    if extraction_manifest
    else "Extraction skipped; using local raw files."
)

### 3.3 Validate the Local Manifest

**Question:** are all seven raw files present, readable and unchanged from the recorded snapshot?

Read Parquet row counts and schema, record byte sizes and SHA-256 fingerprints, and compare against `data/raw/manifest.json`. A mismatch stops analysis rather than silently rewriting the baseline. Sizes displayed below are in MiB.

For the historical files extracted before manifests existed, the first validation records a local baseline with the original extraction time and source reconciliation explicitly unknown. File modification times are not presented as extraction provenance. Future helper-driven exports record timestamps, the source schema and COPY-to-Parquet row reconciliation.

Fingerprints detect subsequent changes; they do not establish the historical source's completeness or semantic quality. The local manifest is ignored by Git together with the raw data.


In [ ]:
# Validate all seven files against the local manifest, without source queries.
raw_manifest = validate_local_snapshot(RAW_DIR)
raw_files = [
    {
        "table": entry["table"],
        "rows": entry["rows"],
        "size_mib": round(entry["size_bytes"] / (1024**2), 2),
    }
    for entry in raw_manifest["files"]
]
pl.DataFrame(raw_files)

## 4. Automated Source Profiling

**Question:** what are the initial structural characteristics of each raw table?

Run DuckDB `SUMMARIZE` over each Parquet file and store the results in `profiles`. The outputs below describe types, ranges, cardinality estimates, distributions and SQL null percentages.

`approx_unique` and quantiles are approximate. Null percentages are rounded and do not capture empty strings, placeholders or missing content inside structured text. Use these profiles to identify questions; use exact, targeted checks to establish keys and coverage.


In [12]:
# Generate an automated profile for every raw table

profiles = {}

for table in TABLES:
    parquet_path = (RAW_DIR / f"{table}.parquet").as_posix()

    profiles[table] = duck.sql(f"""
        SUMMARIZE
        SELECT *
        FROM read_parquet('{parquet_path}');
    """).pl()

    print(f"Profiled: {table}")

Profiled: locations
Profiled: nutritionals
Profiled: prices
Profiled: products
Profiled: weekly_prices
Profiled: weekly_prices_locations
Profiled: weekly_prices_products


### 4.1 Weekly Prices

Inspect the six price-observation fields before the exact grain and temporal checks in section 6.


In [13]:
profiles["weekly_prices"]

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,str,str,str,str,i64,"decimal[9,2]"
"""shop""","""VARCHAR""","""frc""","""plc""",4,null,null,null,null,null,19218071,0.00
"""location""","""VARCHAR""","""anderlecht-veeweyde""","""wilrijk""",15,null,null,null,null,null,19218071,0.00
"""daltix_id""","""VARCHAR""","""0000ba625520cd774f3fc738e27d9d…","""fffeb08682566dedbf69cb8c7fd627…",108487,null,null,null,null,null,19218071,0.00
"""week""","""DATE""","""2019-01-07""","""2020-12-28""",93,"""2020-02-04 04:03:35.545879""",null,"""2019-08-21""","""2020-02-13""","""2020-08-01""",19218071,0.00
"""price""","""DOUBLE""","""0.009000000000000001""","""1538.9""",16789,"""6.250932314132138""","""15.446185810046934""","""2.1010427006280494""","""3.446279652506106""","""6.148778645133943""",19218071,0.00
"""price_promo""","""DOUBLE""","""0.009000000000000001""","""1538.9""",25480,"""6.060071950857532""","""15.088544886594677""","""2.0051180648429177""","""3.3255824749854432""","""5.9666357887525665""",19218071,0.00


### 4.2 Products

Inspect the non-weekly product attributes, including SQL nulls and visible text placeholders.


In [14]:
profiles["products"]

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,i32,i32,i32,i32,i32,i64,"decimal[9,2]"
"""daltix_id""","""VARCHAR""","""000225fe97b2c286a59521c3fd7476…","""fffef4e9004a2be76b93557c5ae430…",31353,null,null,null,null,null,32826,0.00
"""shop""","""VARCHAR""","""frc""","""obmuj""",7,null,null,null,null,null,32826,0.00
"""name""","""VARCHAR""","""#N/A""","""Нak Worteltjes 350g""",38389,null,null,null,null,null,32826,0.12
"""brand""","""VARCHAR""","""""","""нема""",3595,null,null,null,null,null,32826,2.25
"""country""","""VARCHAR""","""be""","""nl""",3,null,null,null,null,null,32826,0.00
"""description""","""VARCHAR""","""""","""﻿rundvleessalade met heerlijk …",25674,null,null,null,null,null,32826,10.34
"""categories""","""VARCHAR""","""[ [ ""2+1 op Snoep"" ] ]""","""[ [ ""zuivel, eieren, bot…",11833,null,null,null,null,null,32826,1.51
"""language""","""VARCHAR""","""fr""","""nl""",2,null,null,null,null,null,32826,0.00


### 4.3 Locations

Inspect non-weekly location attributes. A profile alone cannot establish whether `id` is a valid key.


In [15]:
profiles["locations"]

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,str,str,str,str,i64,"decimal[9,2]"
"""shop""","""VARCHAR""","""frc""","""raps""",10,null,null,null,null,null,1638,0.00
"""country_code""","""VARCHAR""","""be""","""nl""",4,null,null,null,null,null,1638,0.00
"""id""","""VARCHAR""","""0065c""","""ffe44""",1339,null,null,null,null,null,1638,0.00
"""type""","""VARCHAR""",null,null,0,null,null,null,null,null,1638,100.00
"""geolocation_latitude""","""DOUBLE""","""49.5594117""","""59.7139973""",1582,"""50.85913825058744""","""0.39411816770811114""","""50.65655370424243""","""50.87752464216""","""51.09480939058704""",1638,1.10
"""geolocation_longitude""","""DOUBLE""","""2.5924207""","""14.1698445""",1463,"""4.488577900622961""","""0.8197011614008892""","""4.020633977927927""","""4.426998570909091""","""5.069309573242629""",1638,1.10
"""postcode""","""VARCHAR""","""1000""","""9990""",563,null,null,null,null,null,1638,2.44
"""sources""","""VARCHAR""","""[ ""offline"" ]""","""[ ""online"" ]""",3,null,null,null,null,null,1638,0.00


### 4.4 Nutritionals

Inspect the date range and structured nutritional text. Non-null text does not guarantee complete or valid nutrient values.


In [16]:
profiles["nutritionals"]

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,i32,str,str,str,i64,"decimal[9,2]"
"""daltix_id""","""VARCHAR""","""0002c6e437ae0c4f3a9129ba13a702…","""fffffba4674816c2caac0a2e2d913c…",52781,null,null,null,null,null,1096542,0.00
"""shop""","""VARCHAR""","""frc""","""plc""",6,null,null,null,null,null,1096542,0.00
"""country""","""VARCHAR""","""be""","""nl""",2,null,null,null,null,null,1096542,0.00
"""download_date""","""DATE""","""2020-11-27""","""2021-02-24""",90,"""2021-01-01 16:42:41.64415""",null,"""2020-12-11""","""2020-12-26""","""2021-01-28""",1096542,0.00
"""nutritional_values_std""","""VARCHAR""","""{ ""nutrients"": { ""carboh…","""{ ""nutrients"": { ""satura…",46817,null,null,null,null,null,1096542,0.00
"""language""","""VARCHAR""","""nl""","""nl""",1,null,null,null,null,null,1096542,0.00


### 4.5 Prices

Inspect the non-weekly price history. Its promotional field is named `promo_price`, unlike `price_promo` in the weekly table.


In [17]:
profiles["prices"]

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,str,str,str,str,i64,"decimal[9,2]"
"""daltix_id""","""VARCHAR""","""03cfa418d809e4ebce7db71eab220b…","""fc0dc30d90155f84a0cce32cfdda94…",121,null,null,null,null,null,1198547,0.00
"""shop""","""VARCHAR""","""frc""","""plc""",5,null,null,null,null,null,1198547,0.00
"""country""","""VARCHAR""","""be""","""nl""",2,null,null,null,null,null,1198547,0.00
"""location""","""VARCHAR""","""0065c""","""ffe44""",113,null,null,null,null,null,1198547,0.00
"""price""","""DOUBLE""","""0.29700000000000004""","""9.779000000000002""",964,"""2.4645471057878434""","""1.5071818721092347""","""1.39103077389463""","""2.075798574012717""","""3.261665076088472""",1198547,0.00
"""promo_price""","""DOUBLE""","""1.125""","""4.455""",22,"""2.2338778829095007""","""0.6419299210425455""","""1.7009999999999998""","""2.241""","""2.6910000000000003""",1198547,99.44
"""unit_std""","""VARCHAR""","""su""","""su""",1,null,null,null,null,null,1198547,0.00
"""currency""","""VARCHAR""","""eur""","""eur""",1,null,null,null,null,null,1198547,0.00
"""downloaded_on""","""DATE""","""2020-02-25""","""2021-02-25""",371,"""2020-08-26 00:02:44.286924""",null,"""2020-05-29""","""2020-08-26""","""2020-11-23""",1198547,0.00


### 4.6 Weekly Price Locations

Inspect weekly location attributes and their missingness before testing relationships with weekly prices.


In [18]:
profiles["weekly_prices_locations"]

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,str,str,str,str,i64,"decimal[9,2]"
"""shop""","""VARCHAR""","""frc""","""plc""",4,null,null,null,null,null,1230,0.00
"""location""","""VARCHAR""","""a12wilrijk_sg""","""zwijndrecht_sg""",1084,null,null,null,null,null,1230,0.00
"""location_name""","""VARCHAR""","""'s Gravenwezel""","""op-den-Berg""",1170,null,null,null,null,null,1230,0.00
"""shop_type""","""VARCHAR""","""da""","""yxorp""",10,null,null,null,null,null,1230,35.45
"""geolocation_latitude""","""DOUBLE""","""49.5594117""","""59.7139973""",1212,"""50.8547489389824""","""0.4068419485349983""","""50.66309304960937""","""50.87047912761182""","""51.08206745666667""",1230,0.49
"""geolocation_longitude""","""DOUBLE""","""2.5924516""","""14.1698445""",1126,"""4.500730482690436""","""0.8267645891405758""","""4.041201687890625""","""4.427794845783815""","""5.064025354028514""",1230,0.49
"""locality""","""VARCHAR""","""Aalst""","""Étalle""",448,null,null,null,null,null,1230,0.81
"""postcode""","""VARCHAR""","""1000""","""9990""",560,null,null,null,null,null,1230,1.87
"""state""","""VARCHAR""","""Antwerpen""","""Wallonie""",5,null,null,null,null,null,1230,0.81


### 4.7 Weekly Price Products

Inspect weekly product attributes and missingness before testing product keys and join coverage.


In [19]:
profiles["weekly_prices_products"]

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,i32,i32,i32,i32,i32,i64,"decimal[9,2]"
"""daltix_id""","""VARCHAR""","""0000ba625520cd774f3fc738e27d9d…","""fffff2b33a346e37d69d2e978a5fbb…",105208,null,null,null,null,null,114517,0.00
"""shop""","""VARCHAR""","""frc""","""plc""",4,null,null,null,null,null,114517,0.00
"""name""","""VARCHAR""","""""","""żubrówka bison grass vodka 500…",82958,null,null,null,null,null,114517,0.00
"""brand""","""VARCHAR""","""""","""żubrówka""",7850,null,null,null,null,null,114517,5.77
"""country""","""VARCHAR""","""be""","""be""",1,null,null,null,null,null,114517,0.00
"""description""","""VARCHAR""","""""","""﻿ enkel franstalige versie, ne…",84733,null,null,null,null,null,114517,0.00
"""language""","""VARCHAR""","""nl""","""nl""",1,null,null,null,null,null,114517,0.00
"""categories""","""VARCHAR""","""[ [ ""2+1 op Snoep"" ] ]""","""[ [ ""campaign"", ""Win…",5746,null,null,null,null,null,114517,20.34


## 5. Initial Profiling Conclusions

These observations summarize the saved profiles above. They identify questions for semantic assessment; they do not define cleaning rules or certify data quality.

| Table | Rows in saved profile | Main observation | Follow-up |
| --- | ---: | --- | --- |
| `weekly_prices` | 19,218,071 | Six fields report 0.00% SQL nulls; both prices range from approximately 0.009 to 1,538.9. | Exact grain and dates below; promotion meaning and price validity pending. |
| `weekly_prices_products` | 114,517 | `brand` 5.77% null; `categories` 20.34% null; empty text also appears. | Test product keys, semantic missingness and coverage against weekly prices. |
| `weekly_prices_locations` | 1,230 | `shop_type` 35.45% null; coordinates 0.49% null. | Establish the meaning of `location` and test join keys. |
| `prices` | 1,198,547 | `promo_price` 99.44% null; dated 2020-02-25 to 2021-02-25. | Distinguish field availability from promotion incidence; assess its separate grain. |
| `products` | 32,826 | Nulls include name 0.12%, brand 2.25%, description 10.34%, categories 1.51%; empty strings and `#N/A` appear. | Validate identifiers and meaningful completeness. |
| `locations` | 1,638 | `type` is 100% null; some geographic values are missing. | Investigate keys and whether missing coordinates relate to online locations. |
| `nutritionals` | 1,096,542 | Dated 2020-11-27 to 2021-02-24; structured nutritional information is stored as text. | Validate content, repeated records, product coverage and temporal compatibility. |

### Interpretation and working direction

- **Use exact checks for decisions.** The weekly profile's approximate 93 weeks and 108,487 IDs are superseded by the exact counts of 104 weeks and 102,069 IDs in section 6.
- **Do not infer joins from similar counts.** Similar retailer or product cardinalities do not prove matching values, uniqueness or safe join cardinality.
- **Prioritize the weekly group provisionally.** The case discussion identifies weekly data as the main analytical candidate. Keep non-weekly data available for independent assessment and spot checks.
- **Treat nutritionals as optional product enrichment.** Its later date range and repeated observations need investigation before integration with historical prices. A download date is not automatically an effective-from date.
- **Separate observations from business claims.** Null promotional prices do not establish the percentage of products never promoted; filled promotional prices do not prove a promotion occurred.

Earlier location-key checks and the broader architectural decisions are documented in [the project README](../README.md). They are not reproduced by the profiling cells above. No fact/dimension design or cleaning rule has been finalized.


## 6. Weekly Prices — Semantic Assessment

### 6.1 Grain & Uniqueness

**Question:** what does one weekly-price row represent, and do the proposed identifying columns distinguish observations?

Working hypothesis:

> One row represents a product price observation for a specific retailer, location and week.

Proposed identifying combination: `daltix_id + shop + location + week`.

`daltix_id` is treated as a source product identifier in retailer/country context, not a proven universal identifier across retailers. Grain describes the meaning of a row; uniqueness tests whether columns distinguish the stored rows.

#### Distinct combinations at each level

The first query adds retailer, location and week progressively to show how each level changes the number of distinct combinations. Compare the final count with total rows before accepting any unique-key claim.


In [28]:
grain_levels = duck.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT daltix_id) AS unique_daltix_ids,
        COUNT(DISTINCT (daltix_id, shop)) AS unique_daltix_shop,
        COUNT(DISTINCT (daltix_id, shop, location)) AS unique_daltix_shop_location,
        COUNT(DISTINCT (daltix_id, shop, location, week)) AS unique_candidate_grain
    FROM read_parquet('{weekly_prices_path}');
""").pl()

grain_levels

total_rows,unique_daltix_ids,unique_daltix_shop,unique_daltix_shop_location,unique_candidate_grain
i64,i64,i64,i64,i64
19218071,102069,102069,375742,18699187


#### Repeated combinations and price differences

For each proposed weekly combination, count rows and distinct `(price, price_promo)` pairs. Keep groups with more than one row, then summarize them by week.

- `duplicated_grain_combinations`: all repeated groups.
- `exact_duplicate_combinations`: repeated groups with one distinct price pair.
- `same_grain_different_prices`: repeated groups with multiple price pairs.

The last two counts partition the first. The query identifies the pattern; it does not explain its cause or decide which records to retain.


In [27]:
grain_issues = duck.sql(f"""
    WITH grain AS (
        SELECT
            daltix_id,
            shop,
            location,
            week,
            COUNT(*) AS rows_per_grain,
            COUNT(DISTINCT (price, price_promo)) AS distinct_price_pairs
        FROM read_parquet('{weekly_prices_path}')
        GROUP BY ALL
    )

    SELECT
        week,
        COUNT(*) AS duplicated_grain_combinations,
        COUNT(*) FILTER (
            WHERE distinct_price_pairs = 1
        ) AS exact_duplicate_combinations,
        COUNT(*) FILTER (
            WHERE distinct_price_pairs > 1
        ) AS same_grain_different_prices
    FROM grain
    WHERE rows_per_grain > 1
    GROUP BY week
    ORDER BY week;
""").pl()

grain_issues

week,duplicated_grain_combinations,exact_duplicate_combinations,same_grain_different_prices
date,i64,i64,i64
2019-05-27,126619,39196,87423
2019-12-30,188398,47751,140647
2020-12-28,203867,56025,147842


#### Grain conclusion

The saved results show:

| Level | Count |
| --- | ---: |
| Total rows | 19,218,071 |
| Distinct `daltix_id` | 102,069 |
| Distinct `(daltix_id, shop)` | 102,069 |
| Distinct `(daltix_id, shop, location)` | 375,742 |
| Distinct proposed weekly combinations | 18,699,187 |

Adding `shop` does not increase cardinality in this snapshot. This observation does not establish identifier behavior in other tables or future data.

The proposed weekly combination is **not a unique key**. Repeated combinations are concentrated in three weeks:

| Week | Repeated combinations | Identical price pairs | Different price pairs |
| --- | ---: | ---: | ---: |
| 2019-05-27 | 126,619 | 39,196 | 87,423 |
| 2019-12-30 | 188,398 | 47,751 | 140,647 |
| 2020-12-28 | 203,867 | 56,025 | 147,842 |
| **Total** | **518,884** | **142,972** | **375,912** |

These are counts of groups, not counts of rows. Total rows minus distinct proposed combinations is also 518,884; together, the two saved outputs imply that every repeated group contains exactly two rows. Therefore, 1,037,768 rows belong to repeated groups, approximately 5.40% of the dataset.

Where the identifying columns and both prices agree, all six available fields are identical. Where prices differ, the observations are not full-row duplicates. The cause is unresolved: neither a duplicate load nor multiple within-week collections has been established.

**Decision:** retain the grain as a working hypothesis, flag the three weeks for investigation and apply no deduplication or price-selection rule yet.


### 6.2 Temporal Coverage

**Question:** does the overall date range contain the expected number of weekly observations?

The following query measures the first and last dates, the number of distinct dates in `week`, and the expected number of weekly periods between the endpoints.

This is an aggregate check. Equality of observed and expected counts needs calendar alignment to support a no-missing-weeks conclusion, and it does not establish completeness for each product/location series.


In [29]:
temporal_coverage = duck.sql(f"""
    SELECT
        MIN(week) AS first_week,
        MAX(week) AS last_week,
        COUNT(DISTINCT week) AS observed_weeks,
        DATE_DIFF('week', MIN(week), MAX(week)) + 1 AS expected_weeks
    FROM read_parquet('{weekly_prices_path}');
""").pl()

temporal_coverage

first_week,last_week,observed_weeks,expected_weeks
date,date,i64,i64
2019-01-07,2020-12-28,104,104


#### Temporal coverage conclusion

| Metric | Saved result |
| --- | --- |
| First week | 2019-01-07 |
| Last week | 2020-12-28 |
| Observed distinct weeks | 104 |
| Expected weekly periods | 104 |

The saved query is consistent with two years of weekly observations.

**Additional review evidence:** a separate read-only query against the local Parquet confirmed the same endpoints and count, zero null weeks and zero dates outside Monday. Together, those results establish that no weekly dates are missing across the overall range. That supplementary query is not yet included as an executable cell here.

**Limit:** continuity and entry/exit patterns at product/location level remain untested. Aggregate coverage must not be described as complete coverage of every series.


## 7. Next Session

### Resume with promotion logic

The next analytical step is **6.3 Promotion Logic**, to be inserted after temporal coverage when its code is written.

**Question:** how do `price` and `price_promo` relate, and which patterns can defensibly represent a promotion?

Start by measuring nulls and the cases `price_promo < price`, `price_promo = price` and `price_promo > price`. Investigate invalid or non-finite values before deriving discounts. Interpret the results before defining a promotion flag; do not classify every non-null promotional price as a promotion.

### Remaining assessment order

1. Finish weekly-price promotion semantics, price validity and contextual outlier investigation.
2. Assess `weekly_prices_products`: grain, keys, semantic missingness and product coverage.
3. Assess `weekly_prices_locations`: grain, keys and location meaning.
4. Measure weekly joins: matched/unmatched observations, cardinality and row multiplication.
5. Evaluate non-weekly and nutritional data only where relevant to the emerging business question.
6. Define explicit quality rules and justified cleaning decisions before building clean data or marts.

### Starting point for tomorrow

The seven raw files and saved profiling outputs are available. The current grain remains provisional, the three exceptional weeks remain unresolved, and aggregate weekly coverage is established with the supplementary review check described above. No source records have been removed and no analytical joins or promotion rules have been implemented.

Restart the kernel and run the notebook with both source switches set to `False`. The local manifest is validated before profiling. Add the next assessment cells before the cleanup cell, or reopen the local DuckDB connection when continuing after it has closed.


### Session Cleanup

Close the local analytical connection after a complete run. Source connections are already closed by their context managers.


In [ ]:
duck.close()